# Evaluation

Evaluate any model on `tiny_val.txt` and log results to `training_log.csv`.

Set `SOURCE` and `SOURCE_TYPE` in the config cell, then run all cells.

| source_type | source example |
|---|---|
| `"custom"` | `"checkpoints/my-run.pt"` |
| `"hf"` | `"gpt2"` or `"EleutherAI/pythia-70m"` |
| `"fft"` | `"checkpoints/my-fft/"` |
| `"lora"` | `"checkpoints/my-lora/"` |

In [1]:
SOURCE      = "gpt2"
SOURCE_TYPE = "hf"       # "custom" | "hf" | "fft" | "lora"
DEVICE      = "cuda"
VAL_PATH    = "data/tiny_val.txt"
NOTE        = "Untuned gpt2 baseline"

In [2]:
from load import load_model

model, tokenizer = load_model(SOURCE, SOURCE_TYPE, device=DEVICE)
print(f"Loaded {SOURCE_TYPE}: {SOURCE}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded hf: gpt2


In [4]:
from eval import evaluate

# For custom models, also pass cfg (loaded alongside the model)
cfg = model.cfg if SOURCE_TYPE == "custom" else None

tiny_loss = evaluate(model, "data/tiny_val.txt", tokenizer, SOURCE_TYPE, cfg=cfg, device=DEVICE)
full_loss = evaluate(model, "data/full_val.txt", tokenizer, SOURCE_TYPE, cfg=cfg, device=DEVICE)
print(f"tiny loss: {tiny_loss:.4f}")
print(f"full loss: {full_loss:.4f}")

tiny loss: 4.0385
full loss: 4.0062


In [6]:
import math
import types
from utils import log_results

def bpc(loss, text, tokenizer, source_type):
    ids = tokenizer.encode(text).ids if source_type == "custom" else tokenizer.encode(text)
    avg_chars_per_token = len(text) / len(ids)
    return loss / math.log(2) / avg_chars_per_token, avg_chars_per_token

with open("data/tiny_val.txt") as f:
    tiny_text = f.read()
with open("data/full_val.txt") as f:
    full_text = f.read()

tiny_bpc, tiny_cpt = bpc(tiny_loss, tiny_text, tokenizer, SOURCE_TYPE)
full_bpc, full_cpt = bpc(full_loss, full_text, tokenizer, SOURCE_TYPE)
print(f"tiny BPC: {tiny_bpc:.4f} (chars/token: {tiny_cpt:.3f})")
print(f"full BPC: {full_bpc:.4f} (chars/token: {full_cpt:.3f})")

log_cfg = cfg if SOURCE_TYPE == "custom" else types.SimpleNamespace(
    vocab_size=None, layers=None, d_model=None, d_ff=None, H=None,
    B=None, n=None, lr=None, epochs=None
)

log_results(
    cfg=log_cfg,
    train_loss=0.0,
    val_loss=full_loss,
    tiny_val_loss=tiny_loss,
    avg_chars_per_token=tiny_cpt,
    name=SOURCE,
    note=NOTE,
    csv_path="training_log.csv",
)

tiny BPC: 1.8804 (chars/token: 3.098)
full BPC: 1.8363 (chars/token: 3.147)
CSV row appended to training_log.csv
